# spaCy で依存構造パーザを自分で学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/15_parsing/spacy_train_parser.ipynb)

NLP クックブック 15-17 ／ 構文解析

ツリーバンクから依存構造パーザを学習して、精度を確かめたい。自分の分野のデータで学習し直すための手順を知りたい。

In [1]:
!pip install -q ginza ja_ginza

In [2]:
import urllib.request

base = "https://raw.githubusercontent.com/UniversalDependencies/UD_Japanese-GSD/master/ja_gsd-ud-"
for split, n in [("train", 1500), ("dev", 200)]:        # 短時間で終わるように一部だけ使う (全体は train 7050 文)
    sents = urllib.request.urlopen(base + split + ".conllu").read().decode("utf-8").strip().split("\n\n")
    with open(f"{split}.conllu", "w", encoding="utf-8") as f:
        f.write("\n\n".join(sents[:n]) + "\n\n")
    print(split, len(sents), "文中", n, "文を使用")

train 7050 文中 1500 文を使用
dev 507 文中 200 文を使用


In [3]:
# 1. CoNLL-U を spaCy のバイナリ形式に変換  2. 設定ファイルを生成  3. 学習
!python -m spacy convert train.conllu . --converter conllu -n 1 | tail -1
!python -m spacy convert dev.conllu . --converter conllu -n 1 | tail -1
!python -m spacy init config config.cfg --lang ja --pipeline parser --optimize efficiency | tail -1
!python -m spacy train config.cfg --output model --paths.train train.spacy --paths.dev dev.spacy --training.max_steps 600 --training.eval_frequency 200

✔ Generated output file (1500 documents): train.spacy
✔ Generated output file (200 documents): dev.spacy
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy
✔ Created output directory: model
ℹ Saving to output directory: model
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'parser']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS PARSER  DEP_UAS  DEP_LAS  SENTS_F  SCORE 
---  ------  ------------  -----------  -------  -------  -------  ------
  0       0          0.00       219.87    15.52     9.72    17.92    0.13
  0     200       1506.91     12649.75    83.48    76.83    97.28    0.80
  1     400       2045.88      7985.87    86.20    81.15    93.69    0.84
  2     600       2560.98      8119.40    87.57    83.53    98.51    0.86
✔ Saved pipeline to output directory
mode

In [4]:
import spacy

nlp = spacy.load("model/model-best")
doc = nlp("太郎は図書館で花子に借りた本を返した。")
for t in doc:
    print(f"{t.i:2} {t.text:4} {t.dep_:8} → {t.head.text}")

 0 太郎   nsubj    → 返し
 1 は    case     → 太郎
 2 図書   compound → 館
 3 館    obl      → 借り
 4 で    case     → 館
 5 花子   obl      → 借り
 6 に    case     → 花子
 7 借り   acl      → 本
 8 た    aux      → 借り
 9 本    obj      → 返し
10 を    case     → 本
11 返し   ROOT     → 返し
12 た    aux      → 返し
13 。    punct    → 返し


In [5]:
# 学習済みの GiNZA と、同じ開発データで比べる
!python -m spacy evaluate model/model-best dev.spacy | grep -E "UAS|LAS"
!python -m spacy evaluate ja_ginza dev.spacy | grep -E "UAS|LAS"

UAS      87.57
LAS      83.53
=============================== LAS (per type) ===============================
UAS      84.15
LAS      82.38
=============================== LAS (per type) ===============================


- 1500 文・600 ステップ (CPU で 1〜2 分) でも UAS は 80% 台後半になる。全データで十分に学習すれば 90% を超える。
- GiNZA のほうが低く出るのは、GiNZA が別のツリーバンク (UD Japanese-BCCWJ) の基準で学習されていて、単語分割やラベルの付け方が GSD と少し違うため。パーザの優劣ではなく、「評価データと同じ基準で学習したか」の差が出ている。
- `--optimize accuracy` と事前学習済みの Transformer を使う設定 (`--gpu-id 0`、`--pipeline transformer,parser`) にすると精度が大きく上がる。
- 単語分割は spaCy の日本語トークナイザ (SudachiPy) が行う。ツリーバンクの分割と合わない語は学習・評価で対応が取れないので、分割基準の一致が重要。
- 自分の分野のデータを数百文アノテーションして追加学習すると、分野適応の効果が大きい。その場合は既存モデルの重みから始める (`[initialize]` や `source` の設定)。